In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("LocalETL")
    # === MEMORY ALLOCATION ===
    .config("spark.driver.memory", "10g")        # main process (driver)
    .config("spark.executor.memory", "10g")      # worker processes (same JVM locally)
    .config("spark.driver.maxResultSize", "2g")  # prevent result collection errors
    
    # === PARALLELISM ===
    .config("spark.driver.cores", "6")           # use 6 out of 8 cores
    .config("spark.executor.cores", "6")
    .config("spark.default.parallelism", "12")   # usually ~2x num_cores for local
    
    # === PERFORMANCE TUNING ===
    .config("spark.sql.shuffle.partitions", "24")  # parallelize shuffles (joins/groupBy)
    .config("spark.memory.fraction", "0.8")        # 80% of JVM heap for Spark execution
    .config("spark.memory.storageFraction", "0.4") # 40% of that for caching
    .config("spark.sql.files.maxPartitionBytes", "128MB")  # ideal partition size for I/O
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")  # fast Pandas conversion
    .config("spark.local.dir", "/tmp/spark-temp")  # local disk space for shuffle spill
    
    # === OPTIONAL LOGGING + CLEANUP ===
    .config("spark.sql.broadcastTimeout", "600")  # allow large table broadcasts
    .config("spark.cleaner.periodicGC.interval", "5min")  # reduce memory leaks
    
    .getOrCreate()
)


25/10/05 16:08:06 WARN Utils: Your hostname, Parths-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.1.22 instead (on interface en0)
25/10/05 16:08:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/05 16:08:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/10/05 16:08:07 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).
25/10/05 16:08:08 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
from pyspark.sql import functions as F

# --- Example reads ---
transactions_df = spark.read.parquet("data/curated/merchant_transactions/")
consumers_df = spark.read.parquet("data/curated/tbl_consumer/")
fraud_df = spark.read.csv("data/tables/merchant_data/consumer_fraud_probability.csv", header=True, inferSchema=True)
user_df = spark.read.parquet("data/tables/merchant_data/consumer_user_details.parquet")
income_df = spark.read.csv("data/curated/merged_postcode_income.csv", header=True, inferSchema=True)

#Mapping consumer_id to user_id
ctu_df = consumers_df.join(user_df, on="consumer_id", how="left").dropDuplicates(["consumer_id"])

merged_df = (
    transactions_df
    .join(fraud_df, on=["user_id", "order_datetime"], how="left")
    .join(ctu_df, on="user_id", how="left")
    .join(income_df, on="postcode", how="left")

)

merged_df.dropDuplicates(["order_datetime"])
merged_df = merged_df.withColumn("take_rate", F.col("take_rate").cast("float"))


In [3]:
merged_df = merged_df.withColumn("order_day", F.dayofmonth("order_datetime"))
merged_df = merged_df.withColumn("order_month", F.month("order_datetime"))
merged_df = merged_df.withColumn("order_weekday", F.dayofweek("order_datetime"))
merged_df = merged_df.withColumn("order_year", F.year("order_datetime"))

In [4]:
from pyspark.sql.window import Window

window = Window.partitionBy("consumer_id").orderBy("order_datetime")

merged_df = merged_df.withColumn("prev_order_datetime", F.lag("order_datetime").over(window))
merged_df = merged_df.withColumn("days_since_last_order",
                   F.when(F.col("prev_order_datetime").isNotNull(),
                          F.datediff("order_datetime", "prev_order_datetime"))
                   .otherwise(0))

In [5]:
window_consumer = Window.partitionBy("consumer_id")
window_merchant = Window.partitionBy("merchant_abn")

# Consumer-level aggregations
merged_df = merged_df.withColumn("avg_dollar_value_consumer", F.avg("dollar_value").over(window_consumer))
merged_df = merged_df.withColumn("total_transactions_consumer", F.count("dollar_value").over(window_consumer))

# Merchant-level aggregations
merged_df = merged_df.withColumn("avg_dollar_value_merchant", F.avg("dollar_value").over(window_merchant))
merged_df = merged_df.withColumn("total_transactions_merchant", F.count("dollar_value").over(window_merchant))

In [6]:
merged_df = merged_df.withColumn("dollar_take_ratio", F.col("dollar_value") * F.col("take_rate") / 100)
merged_df = merged_df.withColumn("income_dollar_ratio", F.col("median_total_income_2020") / (F.col("dollar_value") + 1))

In [20]:
merged_df = merged_df.withColumn("fraud_label", F.when(F.col("fraud_probability") > 20, 1).otherwise(0))


In [21]:

X = merged_df.filter(F.col("fraud_probability").isNotNull()).filter(F.col("income_bin").isNotNull())
Y = merged_df.filter(F.col("fraud_probability").isNull()).filter(F.col("income_bin").isNotNull())
print(X.count())
print(Y.count())

43795


10577277


In [10]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator

In [11]:
# Example features
numeric_features = ["dollar_value", "take_rate",
    "days_since_last_order", "avg_dollar_value_consumer", "total_transactions_consumer",
    "avg_dollar_value_merchant", "total_transactions_merchant",
    "dollar_take_ratio", "income_dollar_ratio"]
categorical_features = ["rev_band", "state", "income_bin", "postcode", "biz_tags"]

# Target variable
label = "fraud_probability"

In [12]:
# StringIndexer for each categorical feature
indexers = [
    StringIndexer(inputCol=col, outputCol=col + "_idx", handleInvalid="keep")
    for col in categorical_features
]

# OneHotEncoder for each indexed column
encoders = [
    OneHotEncoder(inputCol=col + "_idx", outputCol=col + "_vec")
    for col in categorical_features
]


In [13]:
assembler_inputs = [f + "_vec" for f in categorical_features] + numeric_features

assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features"
)

In [14]:
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol=label,
    numTrees=100,
    maxDepth=10,
    seed=67
)

In [15]:
pipeline = Pipeline(stages=indexers + encoders + [assembler, rf])

In [89]:
train_df, test_df = X.randomSplit([0.8, 0.2], seed=67)

model = pipeline.fit(train_df)
predictions = model.transform(test_df)

25/10/05 15:48:12 WARN DAGScheduler: Broadcasting large task binary with size 1210.8 KiB
25/10/05 15:48:31 WARN DAGScheduler: Broadcasting large task binary with size 1927.6 KiB
25/10/05 15:48:50 WARN DAGScheduler: Broadcasting large task binary with size 3.2 MiB
25/10/05 15:49:12 WARN DAGScheduler: Broadcasting large task binary with size 5.4 MiB
25/10/05 15:49:34 WARN DAGScheduler: Broadcasting large task binary with size 8.7 MiB
25/10/05 15:50:03 WARN DAGScheduler: Broadcasting large task binary with size 13.1 MiB
25/10/05 15:50:32 WARN DAGScheduler: Broadcasting large task binary with size 18.6 MiB
25/10/05 15:51:27 WARN DAGScheduler: Broadcasting large task binary with size 24.1 MiB
25/10/05 15:52:27 WARN DAGScheduler: Broadcasting large task binary with size 24.6 MiB
25/10/05 15:53:17 WARN DAGScheduler: Broadcasting large task binary with size 8.1 MiB
25/10/05 15:53:30 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB


In [16]:
evaluator = RegressionEvaluator(
    labelCol=label,
    predictionCol="prediction",
    metricName="rmse"
)
rmse = evaluator.evaluate(predictions)
r2 = RegressionEvaluator(
    labelCol=label,
    predictionCol="prediction",
    metricName="r2"
).evaluate(predictions)

print(f"RMSE: {rmse:.3f}")
print(f"R²: {r2:.3f}")


NameError: name 'predictions' is not defined

In [22]:
from pyspark.ml.classification import RandomForestClassifier

rf_clf = RandomForestClassifier(
    featuresCol="features",
    labelCol="fraud_label",
    numTrees=100,
    maxDepth=10,
    seed=42
)

In [23]:
pipeline_clf = Pipeline(stages=indexers + encoders + [assembler, rf_clf])

train_df, test_df = X.randomSplit([0.8, 0.2], seed=67)

model_clf = pipeline_clf.fit(train_df)
predictions = model_clf.transform(test_df)

25/10/05 16:26:01 WARN DAGScheduler: Broadcasting large task binary with size 1036.4 KiB
25/10/05 16:26:05 WARN DAGScheduler: Broadcasting large task binary with size 1151.7 KiB
25/10/05 16:26:09 WARN DAGScheduler: Broadcasting large task binary with size 1284.9 KiB


In [19]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol="fraud_label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

roc_auc = evaluator.evaluate(predictions)
print(f"ROC-AUC: {roc_auc:.3f}")


25/10/05 16:17:17 WARN DAGScheduler: Broadcasting large task binary with size 1099.5 KiB


ROC-AUC: 0.659
